# Phase 5: Security Prediction API Test & Validation Notebook
## Project: Adaptive Learning for Cloud Security and Predictive Maintenance

### Objective:
Test and validate the reusable security prediction API (`src/security/predict.py`) for:
1. **Single-Record Mode**: Dict input for individual real-time network traffic records.
2. **Batch Prediction Mode**: CSV / Multi-row DataFrame input for bulk log scanning.
3. **Validation Edge-Cases**: Strict error handling when required features are missing or invalid.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure path to src is available
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if '__file__' in locals() else os.path.abspath('..')
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))

from security.predict import predict_attack

TEST_CSV_PATH = os.path.join(PROJECT_ROOT, 'data', 'security', 'UNSW_NB15_testing-set.csv')
df_raw_test = pd.read_csv(TEST_CSV_PATH)
print(f"Raw Test Data Loaded. Shape: {df_raw_test.shape}")

--- 
## 1. Single-Record Prediction Mode Test
Testing single-record prediction on a **Normal** record and an **Attack** record.

In [ ]:
# Extract a known Normal sample
normal_sample = df_raw_test[df_raw_test['label'] == 0].iloc[0].to_dict()
print("=== TESTING SINGLE-RECORD: NORMAL SAMPLE ===")
res_normal = predict_attack(normal_sample)
print("Prediction Output:", res_normal)

# Extract a known Attack sample
attack_sample = df_raw_test[df_raw_test['label'] == 1].iloc[0].to_dict()
print("\n=== TESTING SINGLE-RECORD: ATTACK SAMPLE ===")
res_attack = predict_attack(attack_sample)
print("Prediction Output:", res_attack)

--- 
## 2. Batch Prediction Mode Test (~20 Rows)
Running batch prediction on a sample of 20 test set records.

In [ ]:
df_batch_sample = df_raw_test.sample(20, random_state=42)
df_batch_results = predict_attack(df_batch_sample)

display_cols = ['proto', 'service', 'state', 'dur', 'sbytes', 'dbytes', 'attack_cat', 'predicted_attack', 'confidence_probability', 'risk_level']
display_cols = [c for c in display_cols if c in df_batch_results.columns]
display(df_batch_results[display_cols].head(10))

--- 
## 3. Validation Edge-Case Test
Verifying that passing invalid/missing features raises a clear `KeyError`.

In [ ]:
invalid_input = {"dur": 0.5, "proto": "tcp"} # Missing required columns
try:
    predict_attack(invalid_input)
except KeyError as e:
    print("✅ Validation Error Caught Successfully:", e)
except Exception as e:
    print("Unexpected error:", type(e), e)

--- 
## 4. Phase 5 Checkpoint Verification

1. **Single-Record Mode**: Works seamlessly, returning `prediction`, `probability`, and temporary `risk_level` placeholder.
2. **Batch Mode**: Works for CSV paths or DataFrames, returning clean predictions and confidence scores.
3. **Validation**: Edge cases handled gracefully with specific `KeyError` messages naming missing columns.